In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import unicodedata


In [6]:
TRAINING_DATA = "/training_data_clean.csv"


def prep_data():
    df = pd.read_csv(TRAINING_DATA)

    # rename
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
        ]

    return df


In [16]:
from typing import Dict, List
from sklearn.compose import ColumnTransformer
 # TODO: can use KNN to find the imputation point instead, in-person imputation
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder

# divide columns by type
text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

ord_categories = [
                '1 — Not at all likely',
                '2 — Unlikely',
                '3 — Neutral / Unsure',
                '4 — Likely',
                '5 — Very likely'
                ]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

def preprocess_text():
    return make_pipeline(
        SimpleImputer(strategy="constant", fill_value ="", missing_values=np.nan)
        # text embedding -> always use llm, tokenize and get embedding vector, using hugging face library (Ex. BERT)
        # return torch tensor, embedding is one of the properties

        # TFIDF manually (classical)
    )

def preprocess_ord():

    return make_pipeline(
        OrdinalEncoder(categories= [ord_categories] * len(ord_cols), handle_unknown='use_encoded_value', unknown_value=np.nan),
        SimpleImputer(strategy="most_frequent")

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    transformers = [
        # ("preprocessing_pipeline_for_free_response_features", preprocess_text(), text_cols),
        ("preprocessing_pipeline_for_ordinal_rating_features", preprocess_ord(), ord_cols),
         ("preprocessing_pipeline_for_categorical_select_features", preprocess_cat(), cat_cols)
    ]

    return ColumnTransformer(transformers=transformers, remainder="drop")


# column_transformer = create_preprocessor()
# column_transformer.set_output(transform="pandas")
# preprocessed_num_cat_features_df = column_transformer.fit_transform(
#     train_df[[NUMERICAL_FEATURE, CATEGORICAL_FEATURE]]
# )

In [17]:
def split_data(df):
    df.columns = [
        'id', 'best_tasks_free', 'acad_tasks_rating', 'best_tasks_select', 'subopt_freq_rating',  'subopt_tasks_select', 'subopt_tasks_free',
        'evidence_freq_rating', 'verify_freq_rating', 'verify_method_free', 'class']
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(students, test_size=0.3, random_state=22)

    X_train_sets = []
    X_test_sets = []

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]


    x_train = train_df.drop(columns=['class'])
    y_train = train_df['class']

    x_test = test_df.drop(columns=['class'])
    y_test = test_df['class']

    curr_preprocessor = create_preprocessor()
    X_train = curr_preprocessor.fit_transform(x_train)
      # only tranfrom to prevent data leakage
    X_test = curr_preprocessor.transform(x_test)

    return X_train



In [21]:
df = prep_data()
X_train = split_data(df)
X_train.shape



/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


(576, 17)